# 3: Hypothesis Testing

Goal: test the strongest EDA patterns with reusable formal outputs. The notebook keeps arrival-level chi square tests for service features and hourly ANOVA comparisons for weather features, while reporting only the test statistic, degrees of freedom, p value, and decision.

**Created files:**

- `outputs/tables/hypothesis_catalog.csv`: canonical list of the hypotheses, analysis units, and test choices used in this notebook.
- `outputs/tables/hypothesis_test_results.csv`: exported formal test results with the statistic, degrees of freedom, p value, and decision for each hypothesis.


## 3.1 Setup and output paths

This section loads the core libraries used in the formal tests, points the notebook to the final merged dataset, and prepares the output folder for the hypothesis tables.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display
from scipy.stats import chi2_contingency, f_oneway

NOTEBOOK_DIR = Path.cwd().resolve()
ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
DATASET_PATH = ROOT / "data" / "processed" / "dataset_final.csv"
OUTPUTS_DIR = ROOT / "outputs"
TABLES_DIR = OUTPUTS_DIR / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")
print(f"Dataset path: {DATASET_PATH.relative_to(ROOT)}")
print(f"Tables directory: {TABLES_DIR.relative_to(ROOT)}")

Project root: /Users/berenaydogan/Documents/4.2/DSA210/Project/DSA210-Project
Dataset path: data/processed/dataset_final.csv
Tables directory: outputs/tables


## 3.2 Load the processed dataset

This section reads the merged dataset and prepares the season field used in the season-based hypothesis.

In [2]:
# Parse every timestamp needed for the calendar features and the hourly weather aggregation.
assert DATASET_PATH.exists(), "Run data_collection.ipynb first so data/processed/dataset_final.csv exists."

date_columns = ["service_date", "scheduled_arrival", "scheduled_arrival_date", "weather_time"]
df = pd.read_csv(DATASET_PATH, parse_dates=date_columns)


## 3.3 Build the samples at the arrival level and the hourly weather level

This section creates the two analysis tables used throughout the notebook. I keep arrival-level records for the service-related chi square tests, and I build one row per weather hour for the ANOVA comparisons on hourly delay rates.

In [3]:
arrival_sample_overview = pd.DataFrame(
    {
        "metric": [
            "rows at the arrival level",
            "late arrivals",
            "on-time-or-early arrivals",
            "delay rate",
            "unique train types",
            "unique lines",
            "unique operators",
            "unique stations",
            "seasons used for H3′",
        ],
        "value": [
            len(df),
            int(df["delayed"].sum()),
            int((1 - df["delayed"]).sum()),
            f"{df['delayed'].mean():.2%}",
            df["train_type"].nunique(),
            df["line_name"].nunique(),
            df["operator"].nunique(),
            df["stop_name"].nunique(),
            ", ".join(df["season"].dropna().astype(str).drop_duplicates().tolist()),
        ],
    }
)

weather_hourly = (
    # Collapse arrivals to one row per matched weather hour so each ANOVA group uses independent hourly summaries.
    df.groupby("weather_time", as_index=False)
    .agg(
        arrival_records=("delayed", "size"),
        delayed_count=("delayed", "sum"),
        temperature_2m=("temperature_2m", "first"),
        precipitation=("precipitation", "first"),
        snowfall=("snowfall", "first"),
        wind_speed_10m=("wind_speed_10m", "first"),
    )
    .sort_values("weather_time")
)
weather_hourly["delay_rate"] = weather_hourly["delayed_count"] / weather_hourly["arrival_records"]
weather_hourly["rainy_hour"] = (weather_hourly["precipitation"] > 0).astype(int)
weather_hourly["snowy_hour"] = (weather_hourly["snowfall"] > 0).astype(int)


def make_quantile_labels(series: pd.Series, prefix: str) -> pd.Series:
    quantile_codes = pd.qcut(series, q=4, labels=False, duplicates="drop")
    return quantile_codes.map(
        lambda value: f"{prefix} quartile {int(value) + 1}" if pd.notna(value) else pd.NA
    )


weather_hourly["temperature_group"] = make_quantile_labels(weather_hourly["temperature_2m"], "Temperature")
weather_hourly["wind_speed_group"] = make_quantile_labels(weather_hourly["wind_speed_10m"], "Wind speed")

weather_sample_overview = pd.DataFrame(
    {
        "metric": [
            "hourly weather observations",
            "hours with rain",
            "hours with snow",
            "mean arrivals per matched hour",
            "median arrivals per matched hour",
            "temperature groups used for H8",
            "wind speed groups used for H11",
        ],
        "value": [
            len(weather_hourly),
            int(weather_hourly["rainy_hour"].sum()),
            int(weather_hourly["snowy_hour"].sum()),
            round(weather_hourly["arrival_records"].mean(), 2),
            round(weather_hourly["arrival_records"].median(), 2),
            weather_hourly["temperature_group"].nunique(dropna=True),
            weather_hourly["wind_speed_group"].nunique(dropna=True),
        ],
    }
)

print(f"Arrival-level sample size (n_arrivals): {len(df):,}")
print(f"Weather-hour sample size (n_weather_hours): {len(weather_hourly):,}")
display(arrival_sample_overview)
display(weather_sample_overview)


Arrival-level sample size (n_arrivals): 90,818
Weather-hour sample size (n_weather_hours): 2,743


,metric,value
0,rows at the arrival level,90818
1,late arrivals,43058
2,on-time-or-early arrivals,47760
3,delay rate,47.41%
4,unique train types,9
5,unique lines,27
6,unique operators,2
7,unique stations,4
8,seasons used for H3′,"Winter, Spring, Summer, Autumn"


,metric,value
0,hourly weather observations,2743.00
1,hours with rain,586.00
2,hours with snow,18.00
3,mean arrivals per matched hour,33.11
4,median arrivals per matched hour,35.00
5,temperature groups used for H8,4.00
6,wind speed groups used for H11,4.00


## 3.4 Hypothesis catalog and test design

This section keeps one canonical hypothesis table for the whole notebook. I use it as the single source of truth for the hypothesis labels, tested variables, analysis units, and test choices, so the execution step stays shorter and there is less risk of the notebook drifting out of sync later.

In [4]:
# Store the hypothesis definitions in one canonical table so the notebook can display, execute, and export them from a single source.
hypothesis_catalog = pd.DataFrame(
    [
        {
            "hypothesis_id": "H1",
            "hypothesis": "Delay rate differs by scheduled arrival hour",
            "feature": "Scheduled arrival hour",
            "tested_variable": "scheduled_arrival_hour",
            "analysis_unit": "arrival",
            "test": "chi_square",
            "operationalization": "24 scheduled-arrival hour categories (0-23)",
            "null_hypothesis": "Delay status is independent of scheduled arrival hour.",
            "alternative_hypothesis": "Delay status differs by scheduled arrival hour.",
        },
        {
            "hypothesis_id": "H2",
            "hypothesis": "Delay rate differs by day of week",
            "feature": "Day of week",
            "tested_variable": "scheduled_arrival_weekday_name",
            "analysis_unit": "arrival",
            "test": "chi_square",
            "operationalization": "7 weekday categories from scheduled arrival timestamps",
            "null_hypothesis": "Delay status is independent of day of week.",
            "alternative_hypothesis": "Delay status differs by day of week.",
        },
        {
            "hypothesis_id": "H3′",
            "hypothesis": "Delay rate differs by season",
            "feature": "Season",
            "tested_variable": "season",
            "analysis_unit": "arrival",
            "test": "chi_square",
            "operationalization": "4 sampled season labels derived from the source month windows",
            "null_hypothesis": "Delay status is independent of season.",
            "alternative_hypothesis": "Delay status differs by season.",
        },
        {
            "hypothesis_id": "H4",
            "hypothesis": "Delay rate differs by weekend status",
            "feature": "Weekend indicator",
            "tested_variable": "is_weekend",
            "analysis_unit": "arrival",
            "test": "chi_square",
            "operationalization": "Binary weekday vs weekend indicator",
            "null_hypothesis": "Delay status is independent of weekend status.",
            "alternative_hypothesis": "Delay status differs by weekend status.",
        },
        {
            "hypothesis_id": "H5",
            "hypothesis": "Delay rate differs by train type",
            "feature": "Train type",
            "tested_variable": "train_type",
            "analysis_unit": "arrival",
            "test": "chi_square",
            "operationalization": "Derived train-type categories from vehicle_text",
            "null_hypothesis": "Delay status is independent of train type.",
            "alternative_hypothesis": "Delay status differs by train type.",
        },
        {
            "hypothesis_id": "H6",
            "hypothesis": "Delay rate differs across line names",
            "feature": "Line name",
            "tested_variable": "line_name",
            "analysis_unit": "arrival",
            "test": "chi_square",
            "operationalization": "Observed line names in the Lausanne area in the 120-day sample",
            "null_hypothesis": "Delay status is independent of line name.",
            "alternative_hypothesis": "Delay status differs across line names.",
        },
        {
            "hypothesis_id": "H7",
            "hypothesis": "Delay rate differs across operators",
            "feature": "Operator",
            "tested_variable": "operator",
            "analysis_unit": "arrival",
            "test": "chi_square",
            "operationalization": "Operator categories present in the filtered dataset",
            "null_hypothesis": "Delay status is independent of operator.",
            "alternative_hypothesis": "Delay status differs across operators.",
        },
        {
            "hypothesis_id": "H8",
            "hypothesis": "Hourly delay rate differs across temperature groups",
            "feature": "Temperature group",
            "tested_variable": "temperature_group",
            "analysis_unit": "weather_hour",
            "test": "one_way_anova",
            "operationalization": "Hourly temperature grouped into sample quartiles; outcome is hourly delay rate",
            "null_hypothesis": "Mean hourly delay rate is the same across temperature groups.",
            "alternative_hypothesis": "Mean hourly delay rate differs across temperature groups.",
        },
        {
            "hypothesis_id": "H9",
            "hypothesis": "Hourly delay rate differs between rainy and dry hours",
            "feature": "Rainy vs dry hour",
            "tested_variable": "rainy_hour",
            "analysis_unit": "weather_hour",
            "test": "one_way_anova",
            "operationalization": "Binary rain indicator at the hourly weather level; outcome is hourly delay rate",
            "null_hypothesis": "Mean hourly delay rate is the same in rainy and dry hours.",
            "alternative_hypothesis": "Mean hourly delay rate differs between rainy and dry hours.",
        },
        {
            "hypothesis_id": "H10",
            "hypothesis": "Hourly delay rate differs between snowy and non-snowy hours",
            "feature": "Snowy vs no-snow hour",
            "tested_variable": "snowy_hour",
            "analysis_unit": "weather_hour",
            "test": "one_way_anova",
            "operationalization": "Binary snow indicator at the hourly weather level; outcome is hourly delay rate",
            "null_hypothesis": "Mean hourly delay rate is the same in snowy and non-snowy hours.",
            "alternative_hypothesis": "Mean hourly delay rate differs between snowy and non-snowy hours.",
        },
        {
            "hypothesis_id": "H11",
            "hypothesis": "Hourly delay rate differs across wind speed groups",
            "feature": "Wind speed group",
            "tested_variable": "wind_speed_group",
            "analysis_unit": "weather_hour",
            "test": "one_way_anova",
            "operationalization": "Hourly wind speed grouped into sample quartiles; outcome is hourly delay rate",
            "null_hypothesis": "Mean hourly delay rate is the same across wind speed groups.",
            "alternative_hypothesis": "Mean hourly delay rate differs across wind speed groups.",
        },
        {
            "hypothesis_id": "H12",
            "hypothesis": "Delay rate differs across stations",
            "feature": "Station",
            "tested_variable": "stop_name",
            "analysis_unit": "arrival",
            "test": "chi_square",
            "operationalization": "4 observed Lausanne-area stations in the filtered dataset",
            "null_hypothesis": "Delay status is independent of station.",
            "alternative_hypothesis": "Delay status differs across stations.",
        },
    ]
)

hypothesis_catalog_display = hypothesis_catalog.assign(
    analysis_unit=hypothesis_catalog["analysis_unit"].replace({"weather_hour": "weather hour"}),
    test=hypothesis_catalog["test"].replace(
        {
            "chi_square": "Chi-square independence test",
            "one_way_anova": "One-way ANOVA",
        }
    ),
)

display(
    hypothesis_catalog_display[
        [
            "hypothesis_id",
            "feature",
            "tested_variable",
            "analysis_unit",
            "test",
            "operationalization",
            "null_hypothesis",
            "alternative_hypothesis",
        ]
    ]
)


,hypothesis_id,feature,tested_variable,analysis_unit,test,operationalization,null_hypothesis,alternative_hypothesis
0,H1,Scheduled arrival hour,scheduled_arrival_hour,arrival,Chi-square independence test,24 scheduled-arrival hour categories (0-23),Delay status is independent of scheduled arriv...,Delay status differs by scheduled arrival hour.
1,H2,Day of week,scheduled_arrival_weekday_name,arrival,Chi-square independence test,7 weekday categories from scheduled arrival ti...,Delay status is independent of day of week.,Delay status differs by day of week.
2,H3′,Season,season,arrival,Chi-square independence test,4 sampled season labels derived from the sourc...,Delay status is independent of season.,Delay status differs by season.
3,H4,Weekend indicator,is_weekend,arrival,Chi-square independence test,Binary weekday vs weekend indicator,Delay status is independent of weekend status.,Delay status differs by weekend status.
4,H5,Train type,train_type,arrival,Chi-square independence test,Derived train-type categories from vehicle_text,Delay status is independent of train type.,Delay status differs by train type.
5,H6,Line name,line_name,arrival,Chi-square independence test,Observed line names in the Lausanne area in th...,Delay status is independent of line name.,Delay status differs across line names.
6,H7,Operator,operator,arrival,Chi-square independence test,Operator categories present in the filtered da...,Delay status is independent of operator.,Delay status differs across operators.
7,H8,Temperature group,temperature_group,weather hour,One-way ANOVA,Hourly temperature grouped into sample quartil...,Mean hourly delay rate is the same across temp...,Mean hourly delay rate differs across temperat...
8,H9,Rainy vs dry hour,rainy_hour,weather hour,One-way ANOVA,Binary rain indicator at the hourly weather le...,Mean hourly delay rate is the same in rainy an...,Mean hourly delay rate differs between rainy a...
9,H10,Snowy vs no-snow hour,snowy_hour,weather hour,One-way ANOVA,Binary snow indicator at the hourly weather le...,Mean hourly delay rate is the same in snowy an...,Mean hourly delay rate differs between snowy a...


## 3.5 Statistical helper functions

This section gathers the reusable chi square and ANOVA helpers so the execution block only has to iterate over the canonical hypothesis table. That keeps the logic shorter and makes later edits much safer.

In [5]:
def run_chi_square_hypothesis(
    data: pd.DataFrame,
    hypothesis_row: dict,
) -> dict:
    contingency_table = pd.crosstab(data[hypothesis_row["tested_variable"]], data["delayed"])
    chi2, p_value, dof, _ = chi2_contingency(contingency_table)
    return {
        "hypothesis_id": hypothesis_row["hypothesis_id"],
        "hypothesis": hypothesis_row["hypothesis"],
        "tested_variable": hypothesis_row["tested_variable"],
        "analysis_unit": hypothesis_row["analysis_unit"],
        "analysis_n": int(contingency_table.to_numpy().sum()),
        "groups": int(contingency_table.shape[0]),
        "test": hypothesis_row["test"],
        "statistic_name": "chi2",
        "statistic": float(chi2),
        "dof": str(dof),
        "p_value": float(p_value),
    }


def run_anova_hypothesis(
    data: pd.DataFrame,
    hypothesis_row: dict,
) -> dict:
    clean_data = data[[hypothesis_row["tested_variable"], "delay_rate"]].dropna().copy()
    grouped_data = [
        group["delay_rate"].reset_index(drop=True)
        for _, group in clean_data.groupby(hypothesis_row["tested_variable"])
    ]
    f_statistic, p_value = f_oneway(*grouped_data)

    return {
        "hypothesis_id": hypothesis_row["hypothesis_id"],
        "hypothesis": hypothesis_row["hypothesis"],
        "tested_variable": hypothesis_row["tested_variable"],
        "analysis_unit": hypothesis_row["analysis_unit"],
        "analysis_n": int(len(clean_data)),
        "groups": int(clean_data[hypothesis_row["tested_variable"]].nunique()),
        "test": hypothesis_row["test"],
        "statistic_name": "F",
        "statistic": float(f_statistic),
        "dof": f"{clean_data[hypothesis_row['tested_variable']].nunique() - 1}, {len(clean_data) - clean_data[hypothesis_row['tested_variable']].nunique()}",
        "p_value": float(p_value),
    }


def run_hypothesis(hypothesis_row: dict) -> dict:
    analysis_datasets = {
        "arrival": df,
        "weather_hour": weather_hourly,
    }
    runners = {
        "chi_square": run_chi_square_hypothesis,
        "one_way_anova": run_anova_hypothesis,
    }
    return runners[hypothesis_row["test"]](analysis_datasets[hypothesis_row["analysis_unit"]], hypothesis_row)

In [6]:
hypothesis_rows = hypothesis_catalog.to_dict("records")

**Analysis units used below**

- `arrival`: one row per train arrival. This unit is used for `H1` through `H7`, where the tested features belong to individual services.
- `weather hour`: one row per matched weather hour. This unit is used for `H8` through `H11`, where many arrivals can share the same hourly weather reading.

We split the analysis this way because the service features naturally live at the arrival level, while the weather comparisons are cleaner after collapsing repeated arrivals that share the same hour-level weather observation.

## 3.6 Execute tests, export tables, and interpret the results

This section runs the full hypothesis family by iterating over the canonical hypothesis table, writes the reusable output tables, and summarizes the test statistic, degrees of freedom, p value, and decision for each hypothesis.

In [7]:
results = [run_hypothesis(hypothesis_row) for hypothesis_row in hypothesis_rows]

**Files produced in next step:** `outputs/tables/hypothesis_test_results.csv` and `outputs/tables/hypothesis_catalog.csv`


In [8]:
results_df = pd.DataFrame(results)
results_df["hypothesis_order"] = results_df["hypothesis_id"].str.extract(r"(\d+)").astype(int)
results_df = results_df.sort_values("hypothesis_order").reset_index(drop=True)
results_df["reject_0_05"] = results_df["p_value"] < 0.05
results_df["decision"] = results_df["reject_0_05"].map(
    {
        True: "Reject null hypothesis",
        False: "Do not reject null hypothesis",
    }
)

results_table = results_df[
    [
        "hypothesis_id",
        "hypothesis",
        "analysis_unit",
        "analysis_n",
        "groups",
        "test",
        "statistic_name",
        "statistic",
        "dof",
        "p_value",
        "reject_0_05",
        "decision",
    ]
].copy()
results_table["test"] = results_table["test"].replace(
    {
        "chi_square": "Chi-square independence test",
        "one_way_anova": "One-way ANOVA",
    }
)
results_table["analysis_unit"] = results_table["analysis_unit"].replace({"weather_hour": "weather hour"})

results_display = results_table.copy()
for column in ["statistic", "p_value"]:
    results_display[column] = results_display[column].map(
        lambda value: f"{value:.4g}" if pd.notna(value) else ""
    )

display(results_display)

results_path = TABLES_DIR / "hypothesis_test_results.csv"
hypothesis_catalog_path = TABLES_DIR / "hypothesis_catalog.csv"
results_table.to_csv(results_path, index=False)
hypothesis_catalog_display.to_csv(hypothesis_catalog_path, index=False)

print(f"Saved results table to {results_path.relative_to(ROOT)}")
print(f"Saved hypothesis catalog to {hypothesis_catalog_path.relative_to(ROOT)}")

,hypothesis_id,hypothesis,analysis_unit,analysis_n,groups,test,statistic_name,statistic,dof,p_value,reject_0_05,decision
0,H1,Delay rate differs by scheduled arrival hour,arrival,90818,24,Chi-square independence test,chi2,1680,23,0,True,Reject null hypothesis
1,H2,Delay rate differs by day of week,arrival,90818,7,Chi-square independence test,chi2,415.9,6,1.07e-86,True,Reject null hypothesis
2,H3′,Delay rate differs by season,arrival,90818,4,Chi-square independence test,chi2,52.37,3,2.501e-11,True,Reject null hypothesis
3,H4,Delay rate differs by weekend status,arrival,90818,2,Chi-square independence test,chi2,369.8,1,2.024e-82,True,Reject null hypothesis
4,H5,Delay rate differs by train type,arrival,90818,9,Chi-square independence test,chi2,3575,8,0,True,Reject null hypothesis
5,H6,Delay rate differs across line names,arrival,90818,27,Chi-square independence test,chi2,4884,26,0,True,Reject null hypothesis
6,H7,Delay rate differs across operators,arrival,90818,2,Chi-square independence test,chi2,1.633,1,0.2014,False,Do not reject null hypothesis
7,H8,Hourly delay rate differs across temperature g...,weather hour,2743,4,One-way ANOVA,F,16.96,"3, 2739",6.538e-11,True,Reject null hypothesis
8,H9,Hourly delay rate differs between rainy and dr...,weather hour,2743,2,One-way ANOVA,F,84.76,"1, 2741",6.485e-20,True,Reject null hypothesis
9,H10,Hourly delay rate differs between snowy and no...,weather hour,2743,2,One-way ANOVA,F,0.03044,"1, 2741",0.8615,False,Do not reject null hypothesis


Saved results table to outputs/tables/hypothesis_test_results.csv
Saved hypothesis catalog to outputs/tables/hypothesis_catalog.csv


## 3.7 Interpretation

This section gives a short summary for the formal results above.

At the 0.05 level, the notebook rejects the following hypotheses:

- `H1`: delay rate differs by scheduled arrival hour.
- `H2`: delay rate differs by day of week.
- `H3′`: delay rate differs by season.
- `H4`: delay rate differs by weekend status.
- `H5`: delay rate differs by train type.
- `H6`: delay rate differs across line names.
- `H8`: hourly delay rate differs across temperature groups.
- `H9`: hourly delay rate differs between rainy and dry hours.
- `H11`: hourly delay rate differs across wind speed groups.
- `H12`: delay rate differs across stations.

The notebook does not reject these hypotheses:

- `H7`: delay rate differs across operators.
- `H10`: hourly delay rate differs between snowy and non-snowy hours.

In this sample, that means the clearest statistical differences appear across timing, service structure, season, station, rain, grouped temperature, and grouped wind conditions, while operator and the snowy versus non-snowy comparison do not show enough evidence here.
